# Step 11 — 3DGS view selection vs GT point cloud

Orbit **three** candidate cameras at **−45° / 0° / +45°** ring azimuth (default), render **RGB+depth** with **gsplat**, score unprojected depth vs a **GT** point cloud, then show **every** candidate and the **winning** view.

**Needs:** CUDA + `gsplat` (same as notebook **03**). Prefer **`sam3d-pipeline` Docker** (`/workspace` = repo root).

## Run All (no path edits)

Use **Run All**. The notebook resolves the AffordSplat tree from **`load_config()`** / env (same rules as training code), draws a random **`Seen/train`** Gaussian row whose matching **`PointCloud/PC_<id>.ply`** exists (retries automatically), runs view selection, and plots results. If nothing is mounted, it falls back to **`examples/gaussian_splat/tiny_gaussians.ply`** with a toy GT derived from splat means.

Optional toggles in the setup cell:

- **`FORCE_TINY_TOY`** — skip real data and always use the tiny example.
- **`RANDOM_SAMPLE_SEED`** — `None` for a fresh draw each run; an `int` for a reproducible sequence of retries.

**Affordance-aware ranking:** when the random row has a **`GS_anno_<id>.ply`**, the objective is **base GT score + `AFFORDANCE_SCORE_WEIGHT` × affordance NN score**.

**Orbit:** default is **world +Y** as the ring pole, then **+90° about +X** (`ORBIT_RING_ROTATION_*`). The toy fallback uses **0°** ring rotation. View selection uses **three** ring azimuths by default (**−45°, 0°, +45°**); set `orbit_azimuth_offsets_deg=None` and `num_views` on `ViewSelectionConfig` for a uniform full-ring sweep again.

## Reconstructed mesh (last cell)

**SAM3D is per object / per splat file.** Each gsplat→SAM3D run (notebook **10**, ``scripts/render_gsplat_and_sam3d.py``, or **batch** ``scripts/batch_gsplat_sam3d.py``) writes ``exports/.../<run>/sam3d_dataset/meta_prerender.json`` listing the input ``GS_*.ply``, then ``<run>/reconstruction/mesh.glb``. The notebook **only** loads a mesh when that metadata matches the **current** splat path (it no longer picks a random newest ``mesh.glb`` from another object).

To build meshes **without** opening notebook **10** first: at the **top of the setup code cell**, set **`MANUAL_RECON_MESH_PATH`** to a `mesh.glb`, or **`MANUAL_SAM3D_RUN_DIR`** to the SAM3D **RUN_DIR** (folder that contains `reconstruction/mesh.glb`), or set **`AUTO_SAM3D_IF_MISSING = True`** to run gsplat→SAM3D in-notebook when no cache exists (**GPU-heavy**). You can also run ``scripts/batch_gsplat_sam3d.py`` from Docker for many ``.ply`` paths.
If there is **no** matching SAM3D run yet, the panel falls back to **AffordSplat ``Mesh/``** next to the Gaussian (when present), else a **convex hull** of the GT point cloud in the **same normalized scene** as the gsplat renders. Matplotlib’s default 3D viewing angle is not the same as a gsplat camera; use the interactive widget to rotate, or set ``view_init`` in the last cell if you want a fixed camera.

To force a mesh path, set **`MANUAL_RECON_MESH_PATH`** or **`MANUAL_SAM3D_RUN_DIR`** at the top of the setup code cell.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


In [ ]:
from pathlib import Path

from datasets.affordsplat_local_dataset import resolve_affordsplat_root, sample_random_affordsplat_row
from rendering.gaussian_ply import load_gaussian_splat_ply
from rendering.gsplat_viewpoint_selection import (
    ViewSelectionConfig,
    apply_scene_normalize,
    load_gt_point_cloud,
    select_best_gsplat_view_for_gt,
)
from reconstruction.gsplat_sam3d_batch import ensure_sam3d_reconstruction_for_splat
from reconstruction.mesh_utils import find_sam3d_reconstruction_mesh_for_splat
from utils.config import load_config

# --- Run All: no path editing. Resolves AffordSplat root from config/env, draws a valid
#     splat + matching PointCloud pair (retries), or falls back to tiny_gaussians + toy GT.
FORCE_TINY_TOY = False  # True = always bundled tiny_gaussians (no Seen/ data)
RANDOM_SAMPLE_SEED = None  # int | None — None = new draw each run; int = reproducible base seed

# --- SAM3D triangle mesh (edit here; no separate terminal required if paths already exist) ---
MANUAL_RECON_MESH_PATH = None  # e.g. ROOT / "exports" / "gsplat_sam3d_runs" / "Seen_train_bag_Gaussian_GS_0017" / "reconstruction" / "mesh.glb"
MANUAL_SAM3D_RUN_DIR = None  # e.g. ROOT / "exports" / "gsplat_sam3d_runs" / "Seen_train_bag_Gaussian_GS_0017"  (parent of reconstruction/)
SAM3D_OUTPUT_ROOT = ROOT / "exports" / "gsplat_sam3d_runs"
AUTO_SAM3D_IF_MISSING = True  # True: run gsplat→SAM3D in this cell when no mesh (GPU + weights; slow)

AFFORDANCE_SCORE_WEIGHT = 0.35
ORBIT_AXIS_MODE = "world"
ORBIT_AXIS = None
ORBIT_RING_ROTATION_DEG = 90.0
ORBIT_RING_ROTATION_AXIS = (1.0, 0.0, 0.0)

OUT_DIR = ROOT / "exports" / "view_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_TINY_TOY = False
RANDOM_ROW = None
affordance_gt_path = None
SPLAT_PLY = None
gt_path = None

RECON_MESH_PATH = None
SAM3D_RUN_DIR = None

cfg_yaml = load_config()
root = resolve_affordsplat_root(cfg_yaml)
if root is not None:
    print("AffordSplat root:", root)

if FORCE_TINY_TOY:
    USE_TINY_TOY = True
elif root is None or not (root / "Seen" / "train").is_dir():
    USE_TINY_TOY = True
    print("No Seen/train under resolved root — using tiny_toy fallback.")
else:
    max_tries = 96
    got = False
    for k in range(max_tries):
        seed_try = None if RANDOM_SAMPLE_SEED is None else int(RANDOM_SAMPLE_SEED) + k
        row = sample_random_affordsplat_row(
            affordsplat_root=root, seed=seed_try, subset="Seen", split="train", cfg=cfg_yaml
        )
        if row is None:
            break
        splat = row.splat_path
        nid = str(row.extras["affordsplat_gaussian_numeric_id"])
        gt = splat.parent.parent / "PointCloud" / f"PC_{nid}.ply"
        if not splat.is_file() or not gt.is_file():
            continue
        RANDOM_ROW = row
        SPLAT_PLY = splat
        gt_path = gt
        affordance_gt_path = row.affordance_gs_anno_path
        if affordance_gt_path is not None and not affordance_gt_path.is_file():
            print("Note: affordance ply missing — affordance term disabled:", affordance_gt_path)
            affordance_gt_path = None
        print("Sample:", row.sample_id)
        print(" splat:", SPLAT_PLY)
        print(" GT:", gt_path)
        if affordance_gt_path is not None:
            print(" affordance:", affordance_gt_path, "| weight:", AFFORDANCE_SCORE_WEIGHT)
        got = True
        break
    if not got:
        USE_TINY_TOY = True
        print("No valid random splat+GT pair after retries — using tiny_toy fallback.")

if USE_TINY_TOY:
    SPLAT_PLY = ROOT / "examples" / "gaussian_splat" / "tiny_gaussians.ply"
    toy = OUT_DIR / "toy_gt_from_splat_means.npy"
    means = load_gaussian_splat_ply(SPLAT_PLY).means
    np.save(toy, means.astype(np.float64))
    gt_path = toy
    RANDOM_ROW = None
    affordance_gt_path = None
    print("tiny_toy | splat:", SPLAT_PLY, "| toy GT:", gt_path)

if not USE_TINY_TOY:

    def _resolve_nb_path(p):
        if p is None:
            return None
        q = Path(p).expanduser()
        return q.resolve() if q.is_absolute() else (ROOT / q).resolve()

    RECON_MESH_PATH = None
    SAM3D_RUN_DIR = None
    m_mesh = _resolve_nb_path(MANUAL_RECON_MESH_PATH)
    m_run = _resolve_nb_path(MANUAL_SAM3D_RUN_DIR)

    if m_mesh is not None:
        if m_mesh.is_file():
            RECON_MESH_PATH = m_mesh
            SAM3D_RUN_DIR = (
                m_run if (m_run is not None and m_run.is_dir()) else RECON_MESH_PATH.parent.parent
            )
            print("Using MANUAL_RECON_MESH_PATH:", RECON_MESH_PATH)
        else:
            print("WARN: MANUAL_RECON_MESH_PATH is not a file — ignoring:", m_mesh)

    if RECON_MESH_PATH is None and m_run is not None and m_run.is_dir():
        cand = m_run / "reconstruction" / "mesh.glb"
        if cand.is_file():
            RECON_MESH_PATH = cand
            SAM3D_RUN_DIR = m_run
            print("Using MANUAL_SAM3D_RUN_DIR:", SAM3D_RUN_DIR)
        else:
            print("WARN: MANUAL_SAM3D_RUN_DIR has no reconstruction/mesh.glb — ignoring:", cand)

    if RECON_MESH_PATH is None:
        RECON_MESH_PATH = find_sam3d_reconstruction_mesh_for_splat(SPLAT_PLY, search_under=ROOT)
        if RECON_MESH_PATH is not None:
            SAM3D_RUN_DIR = RECON_MESH_PATH.parent.parent
            print("SAM3D mesh for this splat (auto):", RECON_MESH_PATH)
        else:
            print(
                "No SAM3D cache for this splat. Set MANUAL_RECON_MESH_PATH / MANUAL_SAM3D_RUN_DIR at the top of this cell, "
                "or AUTO_SAM3D_IF_MISSING=True, or run scripts/batch_gsplat_sam3d.py in Docker."
            )

    if AUTO_SAM3D_IF_MISSING and RECON_MESH_PATH is None:
        _sam = ensure_sam3d_reconstruction_for_splat(
            SPLAT_PLY,
            output_root=SAM3D_OUTPUT_ROOT,
            cfg=cfg_yaml,
            max_points=200_000,
            reference_view_index=0,
        )
        print("SAM3D ensure:", _sam["status"], "|", _sam["run_dir"])
        RECON_MESH_PATH = _sam["mesh_glb"]
        SAM3D_RUN_DIR = _sam["run_dir"]

if SPLAT_PLY is None or gt_path is None:
    raise RuntimeError("Internal error: paths not set")
if not Path(SPLAT_PLY).is_file():
    raise FileNotFoundError(SPLAT_PLY)
if not Path(gt_path).is_file():
    raise FileNotFoundError(gt_path)

_ring_rot_deg = 0.0 if USE_TINY_TOY else ORBIT_RING_ROTATION_DEG

cfg = ViewSelectionConfig(
    image_size=256,
    seed=0,
    orbit_axis=ORBIT_AXIS,
    orbit_axis_mode=ORBIT_AXIS_MODE,
    orbit_ring_rotation_deg=_ring_rot_deg,
    orbit_ring_rotation_axis=ORBIT_RING_ROTATION_AXIS,
    affordance_gt_path=affordance_gt_path,
    affordance_score_weight=AFFORDANCE_SCORE_WEIGHT if affordance_gt_path is not None else 0.0,
)
result = select_best_gsplat_view_for_gt(SPLAT_PLY, gt_path, cfg)
print(
    "Best index:",
    result.best_index,
    "| combined:",
    result.scores[result.best_index],
    "| base:",
    result.base_scores[result.best_index],
    "| aff:",
    result.affordance_scores[result.best_index],
)

_gt_xyz = load_gt_point_cloud(gt_path)
if "scene_normalize_center" in result.meta:
    _sn_c = np.asarray(result.meta["scene_normalize_center"], dtype=np.float64)
    _sn_s = float(result.meta["scene_normalize_scale"])
    _gt_xyz_display = apply_scene_normalize(_gt_xyz, _sn_c, _sn_s)
else:
    _sn_c = np.zeros(3, dtype=np.float64)
    _sn_s = 1.0
    _gt_xyz_display = np.asarray(_gt_xyz, dtype=np.float64)
    print("Note: upgrade rendering/gsplat_viewpoint_selection for scene_normalize_* in meta.")

RUN_LABEL = RANDOM_ROW.sample_id if RANDOM_ROW is not None else ("tiny_toy" if USE_TINY_TOY else "auto")


In [ ]:
n = len(result.all_views)
cols = min(8, n)
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.4 * rows))
axes = np.atleast_1d(axes).ravel()
for i, ax in enumerate(axes):
    if i < n:
        ax.imshow(result.all_views[i].rgb)
        sc = result.scores[i]
        t = str(i)
        _az_im = result.meta.get("view_azimuth_deg")
        if _az_im is not None and i < len(_az_im):
            t += f"\n{_az_im[i]:g}°"
        if not np.isfinite(sc):
            t += "\n(rejected)"
        else:
            t += f"\n{sc:.3g}"
        if i == result.best_index:
            t = "★ " + t
        ax.set_title(t, fontsize=8)
        ax.axis("off")
    else:
        ax.axis("off")
_label = RUN_LABEL
fig.suptitle(f"All sampled views (★ = winner) — {_label}", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(result.best_view.rgb)
sc = result.scores[result.best_index]
_label = RUN_LABEL
_waz = result.meta.get("view_azimuth_deg")
if _waz is not None and result.best_index < len(_waz):
    _wt = f"Winner — {_label}  idx {result.best_index} ({_waz[result.best_index]:g}°)  score={sc:.5g}"
else:
    _wt = f"Winner — {_label}  idx {result.best_index}  score={sc:.5g}"
ax.set_title(_wt, fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import trimesh
from mpl_toolkits.mplot3d import art3d


def _as_trimesh(loaded):
    if isinstance(loaded, trimesh.Scene):
        parts = [
            g
            for g in loaded.geometry.values()
            if isinstance(g, trimesh.Trimesh) and len(g.vertices) and len(g.faces)
        ]
        if not parts:
            return None
        return trimesh.util.concatenate(parts) if len(parts) > 1 else parts[0]
    if isinstance(loaded, trimesh.Trimesh) and len(loaded.faces):
        return loaded
    return None


def _apply_scene_to_mesh(tm: trimesh.Trimesh) -> trimesh.Trimesh:
    """Map mesh vertices into the same normalized scene as gsplat / depth scoring."""
    out = tm.copy()
    v = np.asarray(out.vertices, dtype=np.float64)
    v = (v - _sn_c) * _sn_s
    out.vertices = v
    return out


def _candidate_mesh_paths():
    out = []
    if RECON_MESH_PATH is not None:
        p = Path(RECON_MESH_PATH)
        if p.is_file():
            out.append(p)
    if SAM3D_RUN_DIR is not None:
        p = Path(SAM3D_RUN_DIR) / "reconstruction" / "mesh.glb"
        if p.is_file():
            out.append(p)
    if not USE_TINY_TOY:
        gauss_dir = SPLAT_PLY.parent
        cat_dir = gauss_dir.parent
        m = re.match(r"GS_(\d+)\.ply$", SPLAT_PLY.name, re.I)
        nid = m.group(1) if m else None
        if nid is not None:
            for sub in ("Mesh", "mesh", "Reconstruction", "reconstruction"):
                d = cat_dir / sub
                if not d.is_dir():
                    continue
                for pat in (
                    f"*{nid}*.glb",
                    f"*{nid}*.obj",
                    f"*{nid}*.stl",
                    f"M_{nid}.glb",
                    f"M_{nid}.obj",
                ):
                    for hit in sorted(d.glob(pat)):
                        if hit.is_file():
                            out.append(hit)
    seen = set()
    for p in out:
        p = p.resolve()
        if p not in seen:
            seen.add(p)
            yield p


def _resolve_display_mesh():
    for p in _candidate_mesh_paths():
        try:
            loaded = trimesh.load(p, process=False)
        except Exception as exc:
            print(f"Note: could not load mesh {p}: {type(exc).__name__}: {exc}")
            continue
        tm = _as_trimesh(loaded)
        if tm is not None:
            return _apply_scene_to_mesh(tm), f"Triangle mesh from {p} (aligned to gsplat-normalized scene)"
    vx = np.asarray(_gt_xyz_display, dtype=np.float64)
    if vx.shape[0] >= 4:
        hull = trimesh.PointCloud(vx).convex_hull
        return (
            hull,
            "Proxy mesh: convex hull of GT in the normalized scene (no triangle mesh file found).",
        )
    return None, "No mesh to plot (GT has fewer than 4 points)."


def _axis_set(ax, mesh, pts, *, pad=1.12):
    vs = np.asarray(mesh.vertices, dtype=np.float64)
    bmin = vs.min(axis=0)
    bmax = vs.max(axis=0)
    if pts is not None and len(pts):
        bmin = np.minimum(bmin, pts.min(axis=0))
        bmax = np.maximum(bmax, pts.max(axis=0))
    c = 0.5 * (bmin + bmax)
    span = float((bmax - bmin).max())
    r = max(0.5 * span * pad, 1e-4)
    ax.set_xlim(c[0] - r, c[0] + r)
    ax.set_ylim(c[1] - r, c[1] + r)
    ax.set_zlim(c[2] - r, c[2] + r)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass


def _add_mesh(ax, mesh, *, max_faces=12000, seed=0, **kw):
    faces = np.asarray(mesh.faces)
    if len(faces) > max_faces:
        rng = np.random.default_rng(seed)
        faces = faces[rng.choice(len(faces), max_faces, replace=False)]
    coll = art3d.Poly3DCollection(mesh.vertices[faces], **kw)
    ax.add_collection3d(coll)


mesh_vis, mesh_caption = _resolve_display_mesh()
gtv = np.asarray(_gt_xyz_display, dtype=np.float64)
if mesh_vis is None:
    print(mesh_caption)
else:
    # 1) Mesh alone — axis box from mesh vertices only (its own frame).
    fig1 = plt.figure(figsize=(6.2, 5.2))
    ax_solo = fig1.add_subplot(1, 1, 1, projection="3d")
    _add_mesh(
        ax_solo,
        mesh_vis,
        facecolor="steelblue",
        edgecolor="0.3",
        linewidths=0.05,
        alpha=1.0,
    )
    _axis_set(ax_solo, mesh_vis, None)
    ax_solo.set_title("Reconstructed mesh (mesh-only bounds, normalized scene)")
    ax_solo.view_init(elev=22, azim=-65)
    fig1.text(0.5, 0.02, mesh_caption, ha="center", fontsize=9)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.14)
    plt.show()

    # 2) Comparison with GT — shared bounds so mesh and point cloud fit together.
    fig2 = plt.figure(figsize=(10, 4.8))
    ax0 = fig2.add_subplot(1, 2, 1, projection="3d")
    _add_mesh(
        ax0,
        mesh_vis,
        facecolor="steelblue",
        edgecolor="0.3",
        linewidths=0.05,
        alpha=1.0,
    )
    _axis_set(ax0, mesh_vis, gtv)
    ax0.set_title("Mesh (same scale as GT →)")
    ax0.view_init(elev=22, azim=-65)

    ax1 = fig2.add_subplot(1, 2, 2, projection="3d")
    _add_mesh(
        ax1,
        mesh_vis,
        facecolor="0.75",
        edgecolor="0.3",
        linewidths=0.05,
        alpha=0.35,
    )
    ax1.scatter(gtv[:, 0], gtv[:, 1], gtv[:, 2], s=4, c="darkorange", alpha=0.95, depthshade=False)
    _axis_set(ax1, mesh_vis, gtv)
    ax1.set_title("Mesh + GT point cloud (normalized)")
    ax1.view_init(elev=22, azim=-65)

    fig2.suptitle("Comparison (mesh + GT in a common frame)", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
